# AgentReach AI — Phase 3 Safe Test: One Google Doc only

This notebook tests read-only access to the non-sensitive Google Doc titled `AgentReach Test Document`. It searches for that exact title only. It proceeds only if exactly one native Google Doc matches, then restricts the preview tool to that file ID.

Run the cells in order. Google authorization may grant notebook code broader access than this tool uses, so inspect the code and review the Google permission prompt carefully. Running the approval cell authorizes this notebook to read only the exact-title test document for this test.

This notebook does not list unrelated files, read PDFs/Sheets, or create, edit, share, move, or delete files. If zero or multiple exact-title documents are found, the content-reading and agent tests stop.

In [ ]:
%pip install -q cactus-needle google-api-python-client

## 2. Import dependencies

In [ ]:
import json
import sys
import needle
from google.colab import auth
from googleapiclient.discovery import build

print('Python:', sys.version.split()[0])
print('Needle import: OK')

## 3. Initialize Google Drive access

If Google asks for authorization, review the requested access before approving. Do not continue if you are uncomfortable with the permission prompt.

In [ ]:
auth.authenticate_user()
drive_service = build('drive', 'v3')
print('Drive client initialized.')

## 4. Search for the exact test title only

This query is fixed to the exact title `AgentReach Test Document`. It will not print unrelated personal document names.

In [ ]:
TEST_DOC_NAME = 'AgentReach Test Document'
GOOGLE_DOCS_MIME = 'application/vnd.google-apps.document'
query = f"name = '{TEST_DOC_NAME}' and mimeType = '{GOOGLE_DOCS_MIME}' and trashed = false"
response = drive_service.files().list(
    q=query,
    pageSize=5,
    orderBy='modifiedTime desc',
    fields='files(id,name,mimeType,modifiedTime)',
).execute()
test_docs = response.get('files', [])
print('Exact-title test documents found:', len(test_docs))
if len(test_docs) == 1:
    candidate = test_docs[0]
    print('Exact test document found. Modified:', candidate.get('modifiedTime', 'unknown'))
elif not test_docs:
    print('No matching test document found. Create it first, then rerun this cell and the following cells.')
else:
    print('Multiple documents have this exact title. No document will be read; rename/remove test duplicates manually, then rerun.')

## 5. Select and explicitly approve one test document

Choose a number from the exact-title results above, not an ID from your other files.

In [ ]:
approved_file_id = None
approved_to_read = False
if len(test_docs) == 1:
    chosen_doc = test_docs[0]
    if chosen_doc.get('name') == TEST_DOC_NAME and chosen_doc.get('mimeType') == GOOGLE_DOCS_MIME:
        approved_file_id = chosen_doc['id']
        approved_to_read = True
        print('Approved for this test only: the single exact-title Google Doc.')
        print('No other Drive files will be read by the tool.')
else:
    print('Skipped: exactly one matching test document is required. No content will be read.')

## 6. Define the restricted preview tool

This tool only accepts the approved file ID, verifies that it is a native Google Doc, and returns at most 5,000 characters.

In [ ]:
TOOL_CALL_COUNT = 0

@needle.tool(triggers=[r'(?i)\bpreview approved test doc\b', r'(?i)\bread the approved test document\b'])
def preview_approved_test_doc(file_id: str) -> dict:
    global TOOL_CALL_COUNT
    TOOL_CALL_COUNT += 1
    """Preview up to 5,000 characters from the single Google Doc titled AgentReach Test Document that the user explicitly selected and approved in this notebook. Requires its exact file ID."""
    if not approved_to_read or not approved_file_id:
        return {'ok': False, 'error': 'No test document has been explicitly approved for reading.'}
    if not isinstance(file_id, str) or file_id != approved_file_id:
        return {'ok': False, 'error': 'Access denied: only the exact user-approved test document is allowed.'}
    try:
        meta = drive_service.files().get(
            fileId=approved_file_id,
            fields='id,name,mimeType',
        ).execute()
        if meta.get('mimeType') != GOOGLE_DOCS_MIME or meta.get('name') != TEST_DOC_NAME:
            return {'ok': False, 'error': 'Only the exact-title native Google test document can be previewed.'}
        raw = drive_service.files().export(
            fileId=approved_file_id,
            mimeType='text/plain',
        ).execute()
        text = raw.decode('utf-8', errors='replace') if isinstance(raw, bytes) else str(raw)
        limit = 5000
        return {
            'ok': True,
            'name': meta.get('name'),
            'preview_characters': min(len(text), limit),
            'truncated': len(text) > limit,
            'preview': text[:limit],
            'note': 'Read-only preview of the explicitly approved test document.',
        }
    except Exception as exc:
        return {'ok': False, 'error': f'Test document preview failed: {type(exc).__name__}'}

print('Preview tool defined. Test document approved:', bool(approved_file_id))

## 7. Test the Google Docs API directly

In [ ]:
EXPECTED_SAMPLE_TEXT = 'This is a non-sensitive test document for AgentReach AI. The integration is working.'
if not approved_to_read or not approved_file_id:
    raise AssertionError('No exact-title test document was approved; direct read cannot be verified.')
direct_result = preview_approved_test_doc(approved_file_id)
print(json.dumps(direct_result, indent=2, ensure_ascii=False, default=str))
assert direct_result.get('ok') is True, 'Direct preview failed.'
assert EXPECTED_SAMPLE_TEXT in direct_result.get('preview', ''), 'Expected sample sentence missing from direct preview.'
print('DIRECT GOOGLE DOCS READ: PASS')

## 8. Test Needle 3 tool calling

In [ ]:
EXPECTED_SAMPLE_TEXT = 'This is a non-sensitive test document for AgentReach AI. The integration is working.'
if not approved_to_read or not approved_file_id:
    raise AssertionError('No exact-title test document was approved; Needle test cannot be verified.')
agent = needle.Needle(tools=[preview_approved_test_doc])
calls_before = TOOL_CALL_COUNT
result = agent.run(
    'Call preview_approved_test_doc for this approved file ID: ' + approved_file_id +
    '. Return the exact sample sentence from the document verbatim: ' + EXPECTED_SAMPLE_TEXT
)
serialized_result = json.dumps(result, ensure_ascii=False, default=str)
print(serialized_result)
assert TOOL_CALL_COUNT > calls_before, 'Needle did not invoke preview_approved_test_doc.'
assert EXPECTED_SAMPLE_TEXT in serialized_result, 'Needle result did not contain the expected sample sentence.'
print('NEEDLE TOOL CALL: PASS — tool invocation counted and sample sentence verified.')

## Success checkpoint

- The search finds exactly one native Google Doc with the exact title `AgentReach Test Document`.
- The direct API preview returns `ok: true` and the harmless sample text.
- The Needle result includes the tool execution result for that same document.

No interactive input prompts are used, so you do not need to type confirmation phrases into a Python code cell. If zero or multiple exact-title documents are found, the notebook stops before reading content. If the direct preview succeeds but Needle does not call the tool, Google Docs access works and the remaining issue is tool selection. No writes or deletes are included.